**Student Name:** Noah Pham

**BlazerID:** tpham6


# Part 1: Load and Examine Text


In [1]:
with open("llm_training_corpus.txt") as file:
    # Load the text file
    text = file.read()

    # Print the first 500 characters of the text
    print("First 500 characters of the text:")
    print("=" * 50)
    print(text[:500])
    print("=" * 50)

    # Print the total number of characters in the document
    print("Total number of characters:", len(text))

First 500 characters of the text:
Title: Foundations of Statistical Language Modeling and Natural Language Processing

Section 1: Introduction to Autoregressive Sequence Modeling
Autoregressive language models formulate the probability distribution of a sequence of tokens as the product of conditional probabilities over subsequent tokens. Given an arbitrary token sequence x = (x_1, x_2, ..., x_T), the joint probability P(x) is factorized via the probabilistic chain rule as the product from t=1 to T of P(x_t | x_1, ..., x_{t-1}).
Total number of characters: 40532


## Questions

### 1.1 What text did you use?

The text used is from the file "llm*training_corpus.txt", was created by prompting Google Gemini with this prompt: *"You are a senior machine learning professional with deep experience with large language models. Your goal is to choose a plain-text document containing enough text to generate multiple training sequences. You may use a book, article, or another appropriate .txt document. Return as a .txt file."\_

### 1.2 How many characters are in the document?

40,532 characters


# Part 2: Tokenize the Text


In [2]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
# Tokenize the text
tokens = tokenizer.encode(text)

# Display the first 20 token IDs
print("First 20 token IDs:", tokens[:20])

print("Total number of tokens:", len(tokens))
print("Total number of words (approximate):", len(text.split()))

# Display the decoded text corresponding to the first 20 token IDs
decoded_text = tokenizer.decode(tokens[:20])

print("Decoded text for the first 20 tokens:")
print("=" * 50)
print(decoded_text)
print("=" * 50)

First 20 token IDs: [19160, 25, 4062, 602, 286, 34931, 15417, 9104, 278, 290, 12068, 15417, 28403, 198, 198, 16375, 352, 25, 22395, 284]
Total number of tokens: 8576
Total number of words (approximate): 5351
Decoded text for the first 20 tokens:
Title: Foundations of Statistical Language Modeling and Natural Language Processing

Section 1: Introduction to


## Questions

### 2.1 How many tokens were produced?

8,576 tokens

### 2.2 Is the number of tokens the same as the number of words? Explain why or why not.

No. The number of tokens is not the same as the number of words (8,576 != 5,351) because a token may be a subword.

### 2.3 Select one word from your document that is represented by more than one token. Show the word and its tokens.


In [4]:
found = []
while len(found) < 3:
    words = text.split()
    for word in words:
        word_tokens = tokenizer.encode(word)
        if len(word_tokens) > 1:
            print(f"Word: '{word}' is represented by tokens: {word_tokens}")
            found.append(word)

        if len(found) >= 3:
            break

Word: 'Title:' is represented by tokens: [19160, 25]
Word: 'Foundations' is represented by tokens: [21077, 602]
Word: 'Statistical' is represented by tokens: [17126, 19929]


# Part 3: From Token IDs to Embeddings


In [5]:
import torch

vocab_size = 50_257
embedding_dim = 768

embedding = torch.nn.Embedding(vocab_size, embedding_dim)

/Users/aiinnovationclinic1/Documents/Noah_Workspace/ee-668/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
# Select the first five token IDs and pass them through the embedding layer
token_ids = torch.tensor(tokens[:5])
embedded_tokens = embedding(token_ids)

print("Token IDs:", token_ids[:5])
print("Embedded tokens shape:", embedded_tokens.shape)

Token IDs: tensor([19160,    25,  4062,   602,   286])
Embedded tokens shape: torch.Size([5, 768])


## Questions

### 3.1 What is the shape of the resulting tensor?

The shape is [5, 768], which means it returns 5 elements with 768 dimension.

### 3.2 What does the value 768 represent?

The value 768 represents the dimensionality of the embedding vectors. In this context, each token ID is mapped to a vector of size 768 in the embedding space. This means that for each token, the embedding layer will produce a 768-dimensional vector that captures semantic information about that token.

### 3.3 Why can't a neural network simply treat the token IDs themselves as meaningful numerical values?

Token IDs are just integers that represent specific words in the vocabulary. They don't inherently carry any semantic meaning or relationships between each other. The embedding layer allows the model to learn these relationships and create meaningful representations of each token in a continuous vector space.


# Part 4: Create Input and Target Sequences


In [7]:
max_length = 8
input_sequence = "The quick brown fox jumps over the lazy"
target_sequence = "quick brown fox jumps over the lazy dog"

## Questions

### 4.1 Show your input sequence


In [8]:
tokenized_input = tokenizer.encode(input_sequence)
print("Token IDs for input sequence:", tokenized_input)
print("Decoded text for input sequence:", tokenizer.decode(tokenized_input))

Token IDs for input sequence: [464, 2068, 7586, 21831, 18045, 625, 262, 16931]
Decoded text for input sequence: The quick brown fox jumps over the lazy


### 4.2 Show your target sequence


In [9]:
tokenized_target = tokenizer.encode(target_sequence)
print("Token IDs for target sequence:", tokenized_target)
print("Decoded text for target sequence:", tokenizer.decode(tokenized_target))

Token IDs for target sequence: [24209, 7586, 21831, 18045, 625, 262, 16931, 3290]
Decoded text for target sequence: quick brown fox jumps over the lazy dog


### 4.3 Explain why the target sequence is shifted by one token

The target sequence is shifted by one token to create a causal relationship between the input and output. This allows the model to predict the next token in the sequence based on the previous tokens.

### 4.4 What is the model expected to learn from these input/target pairs?

The model is expected to learn the relationship between the input sequence and the target sequence. Specifically, it should learn to predict the next word or token in a sequence based on the context provided by the input sequence. In this case, given the input "The quick brown fox jumps over the lazy", the model should learn to predict "dog" as the next word in the target sequence. This involves understanding grammar, syntax, and semantic relationships between words in order to generate coherent and contextually appropriate text.


# Part 5: Build a PyTorch Dataset


In [10]:
# Implement the GPT_Dataset class
class GPT_Dataset(torch.utils.data.Dataset):
    def __init__(self, text, tokenizer, max_length=8, stride=8):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.stride = stride

        # Tokenize the text
        self.tokens = tokenizer.encode(text)

        # Create input and target sequences
        self.input_sequences = []
        self.target_sequences = []

        for i in range(0, len(self.tokens) - max_length, stride):
            input_seq = self.tokens[i : i + max_length]
            target_seq = self.tokens[i + 1 : i + max_length + 1]

            self.input_sequences.append(input_seq)
            self.target_sequences.append(target_seq)

    def __len__(self):
        return len(self.input_sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.input_sequences[idx]), torch.tensor(
            self.target_sequences[idx]
        )

## Questions

### 5.1 How many samples did your Dataset create?

It should be the result of `(len(tokens) - max_length) // stride`.

### 5.2 What does `__len__()` return

It returns the number of samples in the dataset.

### 5.3 What does `__getitem__(index)` return

It returns a pair of sample, including the input tokens and the target tokens.


# Part 6: Investigate the Sliding Window


In [11]:
ds_A = GPT_Dataset(text, tokenizer, max_length=8, stride=8)
ds_B = GPT_Dataset(text, tokenizer, max_length=8, stride=4)
ds_C = GPT_Dataset(text, tokenizer, max_length=8, stride=2)

print("Number of samples in ds_A (stride=8):", len(ds_A))
print("Number of samples in ds_B (stride=4):", len(ds_B))
print("Number of samples in ds_C (stride=2):", len(ds_C))

Number of samples in ds_A (stride=8): 1071
Number of samples in ds_B (stride=4): 2142
Number of samples in ds_C (stride=2): 4284


| Experiment | max_length | stride | Number of Samples |
| ---------- | ---------- | ------ | ----------------- |
| A          | 8          | 8      | 1,071             |
| B          | 8          | 4      | 2,142             |
| C          | 8          | 2      | 4,284             |

## Questions

### 6.1 Which experiment produced the most training samples?

Experiment C

### 6.2 Why does decreasing the stride increase the number of samples?

The stride determines how many tokens we move forward in the text to create the next input-target pair. A smaller stride means we take more overlapping sequences, which results in a larger number of samples. Conversely, a larger stride skips more tokens, leading to fewer samples.

### 6.3 Which experiments contain overlapping sequences?

The experiment B and C include overlapping sequences because the stride has smaller value than the max length.

### 6.4 What is one possible advantage of overlapping sequences?

One possible advantage of overlapping sequences is that it allows the model to learn more contextual information from the text. By having overlapping input sequences, the model can capture dependencies and relationships between tokens that may span across different parts of the text. This can lead to better generalization and improved performance on tasks such as language modeling, as the model can leverage the additional context provided by the overlapping sequences.


# Part 7: Create a DataLoader


In [12]:
dataset = GPT_Dataset(text, tokenizer, max_length=8, stride=4)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=False)

In [13]:
input_batch, target_batch = next(iter(dataloader))

print(input_batch)
print(target_batch)
print("Input shape:", input_batch.shape)
print("Target shape:", target_batch.shape)

tensor([[19160,    25,  4062,   602,   286, 34931, 15417,  9104],
        [  286, 34931, 15417,  9104,   278,   290, 12068, 15417],
        [  278,   290, 12068, 15417, 28403,   198,   198, 16375],
        [28403,   198,   198, 16375,   352,    25, 22395,   284]])
tensor([[   25,  4062,   602,   286, 34931, 15417,  9104,   278],
        [34931, 15417,  9104,   278,   290, 12068, 15417, 28403],
        [  290, 12068, 15417, 28403,   198,   198, 16375,   352],
        [  198,   198, 16375,   352,    25, 22395,   284,  5231]])
Input shape: torch.Size([4, 8])
Target shape: torch.Size([4, 8])


## Questions

### 7.1 What is the shape of input_batch?

It is [4, 8].

### 7.2 What is the shape of target_batch?

It is also [4, 8].

### 7.3 What does each dimension represent? For example, if the shape is `torch.Size([4, 8])`, explain what 4 and 8 represent.

The shape `torch.Size([4, 8])` represents a batch of data where:

- The first dimension (4) represents the batch size, meaning there are 4 sequences in this batch.
- The second dimension (8) represents the length of each sequence, meaning each sequence contains 8 tokens.

### 7.4 What is the different between a Dataset and a DataLoader?

A Dataset is a class that provides access to the data, allowing you to retrieve individual samples. It defines how to load and preprocess the data. A DataLoader, on the other hand, is a utility that wraps around a Dataset and provides an iterable over the dataset, allowing for batching, shuffling, and parallel loading of data.


# Part 8: Experiment with Batch Size


In [14]:
def print_batch_shapes(dataset, batch_size):
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=False
    )
    input_batch, target_batch = next(iter(dataloader))
    print(
        "Input and target shapes for batch size",
        batch_size,
        "are",
        input_batch.shape,
        "and",
        target_batch.shape,
    )


print_batch_shapes(dataset, batch_size=1)
print_batch_shapes(dataset, batch_size=2)
print_batch_shapes(dataset, batch_size=4)

Input and target shapes for batch size 1 are torch.Size([1, 8]) and torch.Size([1, 8])
Input and target shapes for batch size 2 are torch.Size([2, 8]) and torch.Size([2, 8])
Input and target shapes for batch size 4 are torch.Size([4, 8]) and torch.Size([4, 8])


| Batch Size | Input Batch Shape | Target Batch Shape |
| ---------- | ----------------- | ------------------ |
| 1          | (1, 8)            | (1, 8)             |
| 2          | (2, 8)            | (2, 8)             |
| 4          | (4, 8)            | (4, 8)             |

## Questions

### 8.1 What changes when the batch size increases?

Both the input batch shape and the target batch shape increase.

### 8.2 Does changing the batch size change the tokenization of the document?

No, it does not.

### 8.3 Does changing the batch size change the individual input/target sequences? Explain.

No, changing the batch size does not change the individual input/target sequences. The batch size only affects how many sequences are grouped together in a single batch during training or evaluation. Each individual input and target sequence remains the same regardless of the batch size; only the number of sequences processed at once changes.


# Part 9


## Questions

### 9.1 Why is tokenization necessary before text can be processed by a neural network?

Tokenization is necessary because neural networks operate on numerical data, and text is inherently in a string format. Tokenization converts the text into a sequence of tokens (words, subwords, or characters), which can then be represented as numerical vectors that the neural network can process.

### 9.2 Why are embeddings used instead of giving token IDs directly to the neural network?

Embeddings are used instead of token IDs because they provide a dense, continuous representation of each token that captures semantic meaning and relationships between tokens. This allows the neural network to learn more meaningful representations of the text, leading to better performance on downstream tasks.

### 9.3 Why does a language-model training Dataset need both an input sequence and a target sequence?

A language model needs both an input sequence and a target sequence to learn the probability distribution over the next word in a sentence. The input sequence is used to predict the target sequence, which is typically the same sequence shifted by one position.

### 9.4 What is the purpose of the DataLoader?

The purpose of the DataLoader is to efficiently load and batch data for training or evaluation. It handles shuffling, batching, and parallel data loading, making it easier to train models on large datasets.

### 9.5 At the end of this assignment, have we actually trained a language model? Explain.

No, we have not actually trained a language model in this assignment. We have prepared the data, tokenized it, created datasets and dataloaders, and explored the structure of the data. However, we have not implemented the training loop or optimized the model parameters based on a loss function. Training a language model would require additional steps such as defining a model architecture, specifying a loss function, and running an optimization algorithm over multiple epochs.
